In [1]:
import re
import statistics

PATH = "corpus_jupyter/Adventures_of_Sherlock_Holmes.txt"

raw = open(PATH, encoding="utf-8").read()

# Drop the Project Gutenberg license boilerplate at the top and bottom.
body = raw.split("*** START OF THE PROJECT GUTENBERG EBOOK", 1)[-1]
body = body.split("*** END OF THE PROJECT GUTENBERG EBOOK", 1)[0]

# Take a passage, then UNWRAP it. Gutenberg hard-wraps lines at ~70 characters,
# but a chunker should reason about logical paragraphs, not arbitrary line breaks.
# So we collapse the in-paragraph newlines and keep blank lines as paragraph marks.
start = body.find("To Sherlock Holmes")
paragraphs = [" ".join(p.split()) for p in body[start:].split("\n\n") if p.strip()]
sample = "\n\n".join(paragraphs[:7])              # first 7 paragraphs of the story

print("whole book :", len(body), "characters")
print("Words in the book:", len(re.findall(r"\w+", body)))
print("our sample :", len(sample), "characters,", sample.count("\n\n") + 1, "paragraphs\n")
print(sample[:300], "...")

whole book : 562250 characters
Words in the book: 105960
our sample : 4081 characters, 7 paragraphs

To Sherlock Holmes she is always _the_ woman. I have seldom heard him mention her under any other name. In his eyes she eclipses and predominates the whole of her sex. It was not that he felt any emotion akin to love for Irene Adler. All emotions, and that one particularly, were abhorrent to his col ...


In [2]:
def show_chunks(chunks, n=3, width=220):
    for i, c in enumerate(chunks[:n]):
        shown = c if len(c) <= width else c[:width] + " ..."
        print(f"--- chunk {i}  (len {len(c)}) " + "-" * 28)
        print(shown.strip())
    if len(chunks) > n:
        print(f"... (+{len(chunks) - n} more chunks)")

def chunk_stats(chunks):
    lens = [len(c) for c in chunks]
    clean = sum(1 for c in chunks if c.rstrip().endswith((".", "!", "?", '"', "”")))
    print(f"chunks         : {len(chunks)}")
    print(f"size  min/avg/max: {min(lens)} / {int(statistics.mean(lens))} / {max(lens)}")
    print(f"clean endings  : {clean}/{len(chunks)} = {100 * clean // len(chunks)}%")

# One knob shared across methods so sizes are comparable.
CHUNK_SIZE = 300

# 1 Fixed size characters, simple chunking.

In [3]:
chunks_fixed = [sample[i:i + CHUNK_SIZE] for i in range(0, len(sample), CHUNK_SIZE)]

show_chunks(chunks_fixed)
print()
chunk_stats(chunks_fixed)

--- chunk 0  (len 300) ----------------------------
To Sherlock Holmes she is always _the_ woman. I have seldom heard him mention her under any other name. In his eyes she eclipses and predominates the whole of her sex. It was not that he felt any emotion akin to love for ...
--- chunk 1  (len 300) ----------------------------
d, precise but admirably balanced mind. He was, I take it, the most perfect reasoning and observing machine that the world has seen, but as a lover he would have placed himself in a false position. He never spoke of the  ...
--- chunk 2  (len 300) ----------------------------
e observer—excellent for drawing the veil from men’s motives and actions. But for the trained reasoner to admit such intrusions into his own delicate and finely adjusted temperament was to introduce a distracting factor  ...
... (+11 more chunks)

chunks         : 14
size  min/avg/max: 181 / 291 / 300
clean endings  : 1/14 = 7%


## 2. Sliding window — fixed size **with overlap**

Same fixed cutting, but each chunk starts a little **before** the previous one
ended. That repeated `overlap` of text means a sentence split across a boundary
still appears whole inside at least one chunk — so a retriever can find it.

The cost: overlap duplicates text, so you store more chunks for the same book.

In [4]:
overlap = 60
step = CHUNK_SIZE - overlap                       # advance less than a full chunk

chunks_window = [sample[i:i + CHUNK_SIZE] for i in range(0, len(sample), step)]

# Show the shared text between two neighbours.
print("tail of chunk 0:", repr(chunks_window[0][-overlap:]))
print("head of chunk 1:", repr(chunks_window[1][:overlap]))
print("--> the overlap is identical, so context survives the cut\n")

show_chunks(chunks_window)
print()
chunk_stats(chunks_window)

tail of chunk 0: 'otions, and that one particularly, were abhorrent to his col'
head of chunk 1: 'otions, and that one particularly, were abhorrent to his col'
--> the overlap is identical, so context survives the cut

--- chunk 0  (len 300) ----------------------------
To Sherlock Holmes she is always _the_ woman. I have seldom heard him mention her under any other name. In his eyes she eclipses and predominates the whole of her sex. It was not that he felt any emotion akin to love for ...
--- chunk 1  (len 300) ----------------------------
otions, and that one particularly, were abhorrent to his cold, precise but admirably balanced mind. He was, I take it, the most perfect reasoning and observing machine that the world has seen, but as a lover he would hav ...
--- chunk 2  (len 300) ----------------------------
a false position. He never spoke of the softer passions, save with a gibe and a sneer. They were admirable things for the observer—excellent for drawing the veil from men’s mo

## 3. Sentence-aware — never cut mid-sentence

Instead of counting characters, we first split the text into **sentences**, then
greedily pack whole sentences into a chunk until adding the next one would blow
the budget. Boundaries now always land between sentences.

To find the sentences we use a plain, readable trick instead of a regex: wherever
a `.`, `!`, or `?` is followed by a space, drop in a line break and split on those
line breaks. The punctuation stays attached, so every sentence keeps its full stop.
(The compact one-line regex that does the same thing is left as a comment in the
code, for anyone who wants it.)

Notice the boundary-quality score jump compared to the naive methods.

In [5]:
# Split the passage into sentences. We keep this deliberately simple and
# regex-free: put a line break after every sentence-ending mark that is followed
# by a space, then split on those line breaks. The punctuation stays attached,
# so each sentence keeps its full stop.
flat = sample.replace("\n", " ")
for end in [". ", "! ", "? "]:
    flat = flat.replace(end, end[0] + "\n")        # ". " -> ".\n", etc.
sentences = [s.strip() for s in flat.split("\n") if s.strip()]

# The exact same result as this one-line regex, kept here for reference:
# sentences = re.split(r"(?<=[.!?])\s+", sample.replace("\n", " "))

chunks_sentence = []
current = ""
for s in sentences:
    if len(current) + len(s) + 1 <= CHUNK_SIZE:
        current = (current + " " + s).strip()
    else:
        if current:
            chunks_sentence.append(current)
        current = s
if current:
    chunks_sentence.append(current)

show_chunks(chunks_sentence)
print()
chunk_stats(chunks_sentence)

--- chunk 0  (len 233) ----------------------------
To Sherlock Holmes she is always _the_ woman. I have seldom heard him mention her under any other name. In his eyes she eclipses and predominates the whole of her sex. It was not that he felt any emotion akin to love for ...
--- chunk 1  (len 263) ----------------------------
All emotions, and that one particularly, were abhorrent to his cold, precise but admirably balanced mind. He was, I take it, the most perfect reasoning and observing machine that the world has seen, but as a lover he wou ...
--- chunk 2  (len 175) ----------------------------
He never spoke of the softer passions, save with a gibe and a sneer. They were admirable things for the observer—excellent for drawing the veil from men’s motives and actions.
... (+14 more chunks)

chunks         : 17
size  min/avg/max: 84 / 238 / 473
clean endings  : 17/17 = 100%


## 4. Recursive — try big separators first, fall back to smaller ones

This is the idea behind LangChain's `RecursiveCharacterTextSplitter`. We keep an
ordered list of separators from coarse to fine:

```
paragraph break  ->  line break  ->  ". "  ->  space  ->  raw characters
```

Split on the **coarsest** separator first. Any piece still larger than the budget
gets re-split using the **next** separator down the list, and so on. The result
respects the largest natural boundary that fits — paragraphs stay whole when they
can, and only genuinely huge blobs get hard-cut.

In [6]:
separators=("\n\n", "\n", ". ", " ", "")
sep, *rest = separators
print(repr(sep), repr(rest))

'\n\n' ['\n', '. ', ' ', '']


In [7]:
#lak;shdl;fkajs;lkdfjaosidfpoansdlknfopiuqwoperbnl;asndl;fkjhaoipsundfl;aknvl;kjhasopdihjropiqwnel;fknasdl;kfhjasoi;dhfl;askhdfo;qwbnl;fnas;kldfhoasdpbfn

In [8]:
def recursive_split(text, size, separators=("\n\n", "\n", ". ", " ", "")):
    if len(text) <= size:
        return [text] if text.strip() else []

    sep, *rest = separators
    if sep == "":                                  # last resort: hard character cut
        return [text[i:i + size] for i in range(0, len(text), size)]

    # Split on this separator but KEEP it attached to each piece, so a sentence
    # never loses its full stop (this is LangChain's keep_separator behaviour).
    parts = text.split(sep)
    parts = [p + sep for p in parts[:-1]] + [parts[-1]]

    chunks, buf = [], ""
    for part in parts:
        if len(buf) + len(part) <= size:
            buf += part                            # keep packing at this level
        elif len(part) > size:                     # one part alone is too big...
            if buf:
                chunks.append(buf)
            chunks.extend(recursive_split(part, size, tuple(rest)))   # ...go finer
            buf = ""
        else:                                      # part fits, but not on top of buf
            if buf:
                chunks.append(buf)
            buf = part
    if buf:
        chunks.append(buf)
    return [c.strip() for c in chunks if c.strip()]

chunks_recursive = recursive_split(sample, CHUNK_SIZE)

show_chunks(chunks_recursive)
print()
chunk_stats(chunks_recursive)

--- chunk 0  (len 233) ----------------------------
To Sherlock Holmes she is always _the_ woman. I have seldom heard him mention her under any other name. In his eyes she eclipses and predominates the whole of her sex. It was not that he felt any emotion akin to love for ...
--- chunk 1  (len 263) ----------------------------
All emotions, and that one particularly, were abhorrent to his cold, precise but admirably balanced mind. He was, I take it, the most perfect reasoning and observing machine that the world has seen, but as a lover he wou ...
--- chunk 2  (len 175) ----------------------------
He never spoke of the softer passions, save with a gibe and a sneer. They were admirable things for the observer—excellent for drawing the veil from men’s motives and actions.
... (+17 more chunks)

chunks         : 20
size  min/avg/max: 27 / 202 / 299
clean endings  : 18/20 = 90%


## 5. Hierarchical — follow the document's own structure

The methods above ignore that a book *already has* structure: chapters, then
paragraphs, then sentences. Hierarchical chunking uses that structure directly.

First we detect the twelve chapter headings (Roman numeral + ALL-CAPS title).
Each chapter becomes a **parent**; its paragraphs become **child** chunks.

In [9]:
chapter_re = re.compile(r"^([IVXLCDM]+)\.[ \t]+(.+)$", re.MULTILINE)
headings = [m for m in chapter_re.finditer(body) if m.group(2).strip().isupper()]

bounds = [m.start() for m in headings] + [len(body)]
chapters = []
for i, m in enumerate(headings):
    title = f"{m.group(1)}. {m.group(2).strip()}"
    content = body[m.start():bounds[i + 1]].strip()
    chapters.append((title, content))

print(f"detected {len(chapters)} chapters:\n")
for title, content in chapters:
    print(f"  {title:46}  {len(content):>6} chars")

detected 12 chapters:

  I. A SCANDAL IN BOHEMIA                          46518 chars
  II. THE RED-HEADED LEAGUE                        49256 chars
  III. A CASE OF IDENTITY                          37916 chars
  IV. THE BOSCOMBE VALLEY MYSTERY                  51353 chars
  V. THE FIVE ORANGE PIPS                          39444 chars
  VI. THE MAN WITH THE TWISTED LIP                 49160 chars
  VII. THE ADVENTURE OF THE BLUE CARBUNCLE         42114 chars
  VIII. THE ADVENTURE OF THE SPECKLED BAND         52949 chars
  IX. THE ADVENTURE OF THE ENGINEER’S THUMB        44602 chars
  X. THE ADVENTURE OF THE NOBLE BACHELOR           44149 chars
  XI. THE ADVENTURE OF THE BERYL CORONET           51001 chars
  XII. THE ADVENTURE OF THE COPPER BEECHES         53136 chars


In paragraph chunking, what happens if information spread across multiple paragraphs?
